In [5]:
# ==========================================
# MULTI-IMAGE UV LEAK DATA COLLECTOR + TRAIN
# ==========================================

import cv2
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier

# -------------------------------
# CONFIG
# -------------------------------
image_folder = "dataset"   # 🔴 PUT ALL IMAGES HERE

image_files = [f for f in os.listdir(image_folder)
               if f.lower().endswith(('.png','.jpg','.jpeg'))]

if len(image_files) == 0:
    raise Exception("No images found in folder")

# -------------------------------
# GLOBAL STORAGE
# -------------------------------
data = []
labels = []
current_label = None
img_index = 0

label_names = {
    0: "Engine Oil",
    1: "Transmission Oil",
    2: "Coolant"
}

# -------------------------------
# LOAD IMAGE FUNCTION
# -------------------------------
def load_image(index):
    path = os.path.join(image_folder, image_files[index])
    img = cv2.imread(path)

    if img is None:
        raise Exception(f"Failed to load {path}")

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return img, hsv

img, hsv_img = load_image(img_index)
display = img.copy()

# -------------------------------
# MOUSE CALLBACK
# -------------------------------
def mouse_callback(event, x, y, flags, param):
    global data, labels, current_label

    if event == cv2.EVENT_LBUTTONDOWN and current_label is not None:
        pixel = hsv_img[y, x]

        h, s, v = int(pixel[0]), int(pixel[1]), int(pixel[2])

        data.append([h, s, v])
        labels.append(current_label)

        print(f"[{len(data)}] HSV={h,s,v} → {label_names[current_label]}")

        cv2.circle(display, (x,y), 5, (0,255,255), -1)

# -------------------------------
# WINDOW SETUP
# -------------------------------
cv2.namedWindow("Collector")
cv2.setMouseCallback("Collector", mouse_callback)

print("""
CONTROLS:
E → Engine Oil
T → Transmission Oil
C → Coolant
N → Next Image
Q → Train & Finish

TIP:
Collect 30–50 points per class across ALL images
""")

# -------------------------------
# MAIN LOOP
# -------------------------------
while True:
    cv2.imshow("Collector", display)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('e'):
        current_label = 0
        print("Label: Engine Oil")

    elif key == ord('t'):
        current_label = 1
        print("Label: Transmission Oil")

    elif key == ord('c'):
        current_label = 2
        print("Label: Coolant")

    elif key == ord('n'):
        img_index += 1

        if img_index >= len(image_files):
            print("No more images")
            img_index = len(image_files) - 1
        else:
            img, hsv_img = load_image(img_index)
            display = img.copy()
            print(f"Switched to: {image_files[img_index]}")

    elif key == ord('q'):
        break

cv2.destroyAllWindows()

# -------------------------------
# TRAIN MODEL
# -------------------------------
if len(data) < 20:
    raise Exception("Collect more samples (at least 20+)")

model = RandomForestClassifier(n_estimators=100)
model.fit(data, labels)

print(f"Model trained with {len(data)} samples")

# -------------------------------
# DETECTION FUNCTIONS
# -------------------------------
def get_mask(hsv):
    lower = np.array([0, 80, 80])
    upper = np.array([179, 255, 255])
    return cv2.inRange(hsv, lower, upper)

def clean_mask(mask):
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    return mask

def get_contours(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return [c for c in contours if cv2.contourArea(c) > 500]

def extract_features(hsv, contour):
    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
    cv2.drawContours(mask, [contour], -1, 255, -1)
    region = hsv[mask == 255]

    if len(region) == 0:
        return [0,0,0]

    return [
        np.mean(region[:,0]),
        np.mean(region[:,1]),
        np.mean(region[:,2])
    ]



CONTROLS:
E → Engine Oil
T → Transmission Oil
C → Coolant
N → Next Image
Q → Train & Finish

TIP:
Collect 30–50 points per class across ALL images

Label: Transmission Oil
[1] HSV=(24, 75, 142) → Transmission Oil
[2] HSV=(22, 109, 159) → Transmission Oil
[3] HSV=(23, 102, 167) → Transmission Oil
[4] HSV=(22, 103, 168) → Transmission Oil
[5] HSV=(23, 104, 142) → Transmission Oil
[6] HSV=(22, 110, 142) → Transmission Oil
[7] HSV=(23, 113, 160) → Transmission Oil
[8] HSV=(23, 111, 154) → Transmission Oil
[9] HSV=(22, 95, 156) → Transmission Oil
[10] HSV=(22, 91, 180) → Transmission Oil
[11] HSV=(22, 86, 180) → Transmission Oil
[12] HSV=(20, 81, 177) → Transmission Oil
[13] HSV=(22, 100, 168) → Transmission Oil
[14] HSV=(22, 104, 167) → Transmission Oil
[15] HSV=(22, 96, 154) → Transmission Oil
Switched to: WhatsApp Image 2026-03-25 at 1.57.51 PM.jpeg
Label: Engine Oil
[16] HSV=(90, 7, 255) → Engine Oil
[17] HSV=(79, 24, 255) → Engine Oil
[18] HSV=(83, 18, 255) → Engine Oil
[19] HSV=(83, 

In [2]:

# -------------------------------
# TEST ON LAST IMAGE
# -------------------------------
test_img = img.copy()
hsv = cv2.cvtColor(test_img, cv2.COLOR_BGR2HSV)

mask = clean_mask(get_mask(hsv))
contours = get_contours(mask)

for cnt in contours:
    x, y, w, h = cv2.boundingRect(cnt)

    features = extract_features(hsv, cnt)
    pred = model.predict([features])[0]
    conf = np.max(model.predict_proba([features]))

    if pred == 0:
        color = (255,0,0)
    elif pred == 1:
        color = (0,165,255)
    else:
        color = (0,255,0)

    text = f"{label_names[pred]} ({conf:.2f})"

    cv2.rectangle(test_img, (x,y), (x+w, y+h), color, 2)
    cv2.putText(test_img, text, (x,y-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

cv2.imshow("RESULT", test_img)
cv2.imshow("MASK", mask)

cv2.waitKey(0)
cv2.destroyAllWindows()


NameError: name 'img' is not defined